# B2-019 — Session 4: Attention Module and Tiny Training

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).


## 1. Module contract

A minimal module owns four linear maps: query, key, value, and output. It validates rank 3, shared batch/sequence shapes, positive width, divisibility by heads, a floating dtype, finite inputs, and Boolean mask broadcastability. These checks make shape mistakes fail at the boundary instead of producing a plausible square tensor later.

**Worked example 1.** An input `(2,5,12)` with three heads has head width four. An input width 10 with three heads must be rejected rather than truncated.

**Checkpoint 1A.** Which projection widths must agree before score multiplication?

**Checkpoint 1B.** Why store the scale as $d_h^{-1/2}$?

In [ ]:
import math

import torch
from torch import nn

SEED = 20260808
torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
assert DEVICE.type == "cpu"

## 2. Forward pass and mask

Project, split heads, compute scaled scores, apply the Boolean mask before softmax, multiply by values, concatenate, then apply the output projection. Softmax acts on the final key axis. Reject a broadcast row with no allowed key because all-$-\infty$ logits would not define a probability distribution. Test exact shapes, row sums, finiteness, and forbidden weights equal to zero.

**Worked example 2.** An identity-projection single-head module must reproduce the NumPy computation from Session 2 to the stated tolerance. This checks the implementation against an independent representation.

**Checkpoint 2A.** Which dimension receives softmax?

**Checkpoint 2B.** What mask dtype does this unit require?

In [ ]:
class CausalSelfAttention(nn.Module):
    """One-head causal attention with explicit projections and mask semantics."""

    def __init__(self, width):
        super().__init__()
        self.q = nn.Linear(width, width, bias=False, dtype=torch.float64)
        self.k = nn.Linear(width, width, bias=False, dtype=torch.float64)
        self.v = nn.Linear(width, width, bias=False, dtype=torch.float64)
        self.out = nn.Linear(width, width, bias=False, dtype=torch.float64)

    def forward(self, x):
        if x.ndim != 3:
            raise ValueError("x must have shape (batch, sequence, width)")
        _, length, width = x.shape
        q, k, v = self.q(x), self.k(x), self.v(x)
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(width)
        allowed = torch.tril(
            torch.ones(length, length, dtype=torch.bool, device=x.device)
        )
        weights = torch.softmax(scores.masked_fill(~allowed, float("-inf")), dim=-1)
        return self.out(torch.matmul(weights, v)), weights


PROBE_INPUT = torch.tensor(
    [[[1.0, 0.0, 0.0, 0.0],
      [0.0, 1.0, 0.0, 0.0],
      [0.0, 0.0, 1.0, 0.0]]],
    dtype=torch.float64,
    device=DEVICE,
)
attention_probe = CausalSelfAttention(width=4).to(DEVICE)
with torch.no_grad():
    for layer in (attention_probe.q, attention_probe.k, attention_probe.v, attention_probe.out):
        layer.weight.copy_(torch.eye(4, dtype=torch.float64))
attention_probe_output, attention_probe_weights = attention_probe(PROBE_INPUT)
assert attention_probe_output.shape == (1, 3, 4)
assert torch.count_nonzero(attention_probe_weights.triu(diagonal=1)).item() == 0

## 3. Seeded causal prediction task

Inputs are a pinned one-hot tensor with positional signals added numerically; no learned embedding lookup is needed. At token $i$, the target is the next token class. A causal mask prevents the hidden state at $i$ from reading future input positions, so the target cannot leak through attention. The classifier maps each hidden row to class logits.

**Checkpoint 3A.** Why shift inputs and targets by one position?

**Checkpoint 3B.** Which rows contribute to cross-entropy?

In [ ]:
TOKENS = torch.tensor([0, 1, 2, 1, 3, 0], dtype=torch.long, device=DEVICE)
ONE_HOT = nn.functional.one_hot(TOKENS, num_classes=4).to(dtype=torch.float64)
positions = torch.arange(5, dtype=torch.float64, device=DEVICE).unsqueeze(1)
rates = torch.tensor([1.0, 0.1], dtype=torch.float64, device=DEVICE).unsqueeze(0)
POSITIONAL = torch.empty(5, 4, dtype=torch.float64, device=DEVICE)
POSITIONAL[:, 0::2] = torch.sin(positions * rates)
POSITIONAL[:, 1::2] = torch.cos(positions * rates)
PINNED_INPUTS = (ONE_HOT[:-1] + 0.1 * POSITIONAL).unsqueeze(0)
PINNED_TARGETS = TOKENS[1:].unsqueeze(0)
PINNED_ALLOWED = torch.tril(torch.ones(5, 5, dtype=torch.bool, device=DEVICE))
assert PINNED_INPUTS.shape == (1, 5, 4)
assert PINNED_TARGETS.shape == (1, 5)


class TinyCausalPredictor(nn.Module):
    def __init__(self, width, classes):
        super().__init__()
        self.attention = CausalSelfAttention(width)
        self.classifier = nn.Linear(
            width, classes, bias=True, dtype=torch.float64
        )

    def forward(self, x):
        hidden, _ = self.attention(x)
        return self.classifier(hidden)

## 4. Training loop

Set seed 20260808, instantiate on CPU, compute logits, call cross-entropy on flattened logits and integer targets, then run `zero_grad`, `backward`, and `optimizer.step`. Cross-entropy consumes raw logits and performs its own stable log-softmax. Record a loss trace and deterministic probe logits before and after training.

**Worked example 3.** A uniform three-class predictor has loss $\log 3$ per row. A lower final loss alone is insufficient evidence if every parameter stayed fixed.

**Checkpoint 4A.** Should softmax be called before cross-entropy?

**Checkpoint 4B.** What makes the experiment reproducible?

In [ ]:
torch.manual_seed(SEED)
model = TinyCausalPredictor(width=4, classes=4).to(DEVICE)
initial_parameters = torch.cat(
    [parameter.detach().reshape(-1) for parameter in model.parameters()]
).clone()
optimizer = torch.optim.Adam(model.parameters(), lr=0.04)
loss_trace = []

for _ in range(30):
    optimizer.zero_grad(set_to_none=True)
    logits = model(PINNED_INPUTS)
    loss = nn.functional.cross_entropy(
        logits.reshape(-1, 4), PINNED_TARGETS.reshape(-1)
    )
    loss_trace.append(float(loss.detach()))
    loss.backward()
    optimizer.step()

with torch.no_grad():
    final_logits = model(PINNED_INPUTS)
    first_probe = final_logits[0, 0].clone()
    last_probe = final_logits[0, -1].clone()
    final_parameters = torch.cat(
        [parameter.reshape(-1) for parameter in model.parameters()]
    )
    parameter_delta = float(
        torch.linalg.vector_norm(final_parameters - initial_parameters)
    )

EXPECTED_FIRST_PROBE = [
    -13.715264510619917, 10.189913295038819,
    2.429452795078639, -10.753664065585196,
]
EXPECTED_LAST_PROBE = [
    12.806249447394833, -10.08909666880871,
    -1.0990388127109036, 10.654088871967703,
]
assert len(loss_trace) == 30 and loss_trace[-1] < loss_trace[0]
assert parameter_delta > 0.0
assert torch.allclose(
    first_probe,
    torch.tensor(EXPECTED_FIRST_PROBE, dtype=torch.float64),
    atol=1e-10,
    rtol=1e-10,
)
assert torch.allclose(
    last_probe,
    torch.tensor(EXPECTED_LAST_PROBE, dtype=torch.float64),
    atol=1e-10,
    rtol=1e-10,
)

## 5. Gradient and parameter-update audit

A credible training check records a parameter snapshot, computes one scalar loss through the live graph, verifies finite nonzero gradients after `backward`, performs the optimizer step, and measures a nonzero parameter update. The deterministic run also checks that the loss trace decreases and that pinned probe logits match expected values. Each assertion tests a different link: graph connectivity, gradient calculation, optimizer wiring, and reproducibility.

**Worked example 4.** If loss decreases but `parameter_delta == 0`, the trace may have been computed from different inputs rather than learned parameters; reject the run.

**Checkpoint 5A.** Why check both a gradient and a parameter delta?

**Checkpoint 5B.** Why pin probe logits as well as the loss direction?

## 6. Common pitfalls

Broken: build a float mask but interpret True as forbidden in one place and allowed in another. Fix: name it `allowed` and test corner cells. Broken: detach logits before loss. Fix: keep the computation graph through the scalar loss. Broken: call `optimizer.step()` without first clearing stale gradients. Fix: use the complete update order and audit it.

**Exam connections.** From-scratch tasks grade scale, mask placement, shape contract, gradient flow, and actual parameter update separately.

**Going deeper.** Session 5 wraps attention with residual and feed-forward sublayers.

Checkpoint answers: 1A query/key head width; 1B stable explicit scale; 2A key axis; 2B Boolean; 3A next-step supervision; 3B nonpadding prediction positions; 4A no; 4B fixed inputs, seed, CPU, APIs, and tolerances; 5A they test graph flow and optimizer wiring separately; 5B many wrong runs can still decrease loss.